### Comparación de varias medias: ANOVA

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import col, when, datediff, max as spark_max, min as spark_min, count, sum as spark_sum, avg, desc, asc
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [2]:
spark = SparkSession.builder \
    .appName("LRFM_HYM") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128MB") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

In [3]:
# 1. Cargar datos y crear vista temporal
df = spark.read.parquet('merge_pyspark')
df.createOrReplaceTempView("customer_transactions")

In [4]:
# Exploramos la estructura del dataset
print("=== ESTRUCTURA DEL DATASET ===")
df.printSchema()
print(f"\nTotal de registros: {df.count():,}")

=== ESTRUCTURA DEL DATASET ===
root
 |-- customer_id: string (nullable = true)
 |-- article_id: long (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: long (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullab

In [5]:
#Head 5
print("\n=== HEAD 5 REGISTROS ===")
df.limit(5).toPandas()


=== HEAD 5 REGISTROS ===


,customer_id,article_id,Fecha,price,sales_channel_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,...,section_name,garment_group_no,garment_group_name,detail_desc,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00030bc026428965d1868e4f3536f131feeedf5ce1b003...,673677015,2019-11-28,0.025407,2,673677,Henry polo (1),252,Sweater,Garment Upper body,...,Womens Tailoring,1003,Knitwear,"Jumper in a soft, fine knit with a ribbed polo...",NaN,NaN,ACTIVE,NONE,57,2c29ae653a9282cce4151bd87643c907644e09541abc28...
1,00030bc026428965d1868e4f3536f131feeedf5ce1b003...,673677010,2019-11-28,0.025407,2,673677,Henry polo (1),252,Sweater,Garment Upper body,...,Womens Tailoring,1003,Knitwear,"Jumper in a soft, fine knit with a ribbed polo...",NaN,NaN,ACTIVE,NONE,57,2c29ae653a9282cce4151bd87643c907644e09541abc28...
2,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...,683001006,2019-02-10,0.016932,2,683001,Pattern 7p Socks,302,Socks,Socks & Tights,...,"Womens Nightwear, Socks & Tigh",1021,Socks and Tights,Fine-knit socks in a soft cotton blend.,NaN,NaN,ACTIVE,NONE,29,24e3594738f327e8a7671ec6d1e18b308fb0282e1f7e23...
3,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...,568571002,2018-10-24,0.013542,2,568571,Spartacus cheeky hipster,59,Swimwear bottom,Swimwear,...,"Womens Swimwear, beachwear",1018,Swimwear,"Fully lined, textured bikini bottoms with a lo...",NaN,NaN,ACTIVE,NONE,29,24e3594738f327e8a7671ec6d1e18b308fb0282e1f7e23...
4,00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4...,723469007,2019-02-10,0.025407,2,723469,Kelly Push (Melbourne) ctn 2p,306,Bra,Underwear,...,Womens Lingerie,1017,"Under-, Nightwear",Push-up bras in soft cotton jersey with lace. ...,NaN,NaN,ACTIVE,NONE,29,24e3594738f327e8a7671ec6d1e18b308fb0282e1f7e23...


### ANOVA de un factor sobre HyM (price ~ product_group_name)

- **Paso 1**: Dividir por periodo de fechas especificado.
- **Paso 2**: ANOVA de un factor (comparación de varias medias) usando `product_group_name` como factor y `price` como variable respuesta.
- Se emplean agregaciones en Spark (sin muestreo) para calcular sumas de cuadrados y el estadístico F con su p-valor.


In [ ]:
#  2 conjuntos por rango de fechas
from pyspark.sql.functions import to_date

_df = df.withColumn("Fecha", col("Fecha").cast("date"))

conjunto1 = _df.filter((col("Fecha") >= F.lit("2018-09-20").cast("date")) & (col("Fecha") <= F.lit("2019-12-31").cast("date")))
conjunto2 = _df.filter((col("Fecha") >= F.lit("2020-01-01").cast("date")) & (col("Fecha") <= F.lit("2020-09-22").cast("date")))

print("Registros por conjunto:")
print("Conjunto 1:", f"{conjunto1.count():,}")
print("Conjunto 2:", f"{conjunto2.count():,}")


Registros por conjunto:
Conjunto 1: 20,808,192
Conjunto 2: 10,980,132


In [ ]:
#  ANOVA de un factor (price ~ product_group_name) en cada conjunto
from pyspark.sql import DataFrame
from pyspark.sql.functions import count as spark_count, mean as spark_mean
from pyspark.sql.functions import sum as spark_sum

try:
    from scipy.stats import f as scipy_f_dist
    _has_scipy = True
except Exception:
    _has_scipy = False

import math


def anova_oneway_spark(input_df: DataFrame, group_col: str, value_col: str):
    # Métricas globales
    global_stats = input_df.agg(
        spark_count(value_col).alias("N"),
        spark_mean(value_col).alias("grand_mean"),
        spark_sum((col(value_col) * col(value_col))).alias("sum_x2")
    ).collect()[0]
    N = int(global_stats["N"]) if global_stats["N"] is not None else 0
    grand_mean = float(global_stats["grand_mean"]) if global_stats["grand_mean"] is not None else float("nan")

    if N == 0 or math.isnan(grand_mean):
        return {
            "k": 0,
            "N": N,
            "SSB": float("nan"),
            "SSW": float("nan"),
            "MSB": float("nan"),
            "MSW": float("nan"),
            "df_between": 0,
            "df_within": 0,
            "F": float("nan"),
            "p_value": float("nan"),
        }

    # Métricas por grupo: n_g, mean_g, sum_x2_g
    by_group = (
        input_df.groupBy(group_col)
        .agg(
            spark_count(value_col).alias("n_g"),
            spark_mean(value_col).alias("mean_g"),
            spark_sum(col(value_col) * col(value_col)).alias("sum_x2_g"),
        )
    )

    group_rows = by_group.collect()
    k = len(group_rows)

    # SSB y SSW a partir de agregados
    SSB = 0.0
    SSW = 0.0
    for row in group_rows:
        n_g = int(row["n_g"]) if row["n_g"] is not None else 0
        mean_g = float(row["mean_g"]) if row["mean_g"] is not None else 0.0
        sum_x2_g = float(row["sum_x2_g"]) if row["sum_x2_g"] is not None else 0.0
        SSB += n_g * (mean_g - grand_mean) ** 2
        SSW += sum_x2_g - n_g * (mean_g ** 2)

    df_between = max(k - 1, 0)
    df_within = max(N - k, 1)

    MSB = float("nan")
    MSW = float("nan")
    F_stat = float("nan")

    if df_between > 0 and df_within > 0 and SSW > 0:
        MSB = SSB / df_between
        MSW = SSW / df_within
        if MSW > 0:
            F_stat = MSB / MSW

    # p-valor (upper tail)
    if _has_scipy and math.isfinite(F_stat):
        try:
            p_value = float(1.0 - scipy_f_dist.cdf(F_stat, df_between, df_within))
        except Exception:
            p_value = float("nan")
    else:
        p_value = float("nan")

    return {
        "k": k,
        "N": N,
        "SSB": SSB,
        "SSW": SSW,
        "MSB": MSB,
        "MSW": MSW,
        "df_between": df_between,
        "df_within": df_within,
        "F": F_stat,
        "p_value": p_value,
    }

res1 = anova_oneway_spark(conjunto1.select("product_group_name", "price").na.drop(subset=["product_group_name", "price"]), "product_group_name", "price")
res2 = anova_oneway_spark(conjunto2.select("product_group_name", "price").na.drop(subset=["product_group_name", "price"]), "product_group_name", "price")

print("ANOVA Conjunto 1 (2018-09-20 a 2019-12-31)")
print(res1)
print("\nANOVA Conjunto 2 (2020-01-01 a 2020-09-22)")
print(res2)


ANOVA Conjunto 1 (2018-09-20 a 2019-12-31)
{'k': 17, 'N': 20808192, 'SSB': 769.8969162970706, 'SSW': 7621.552932937875, 'MSB': 48.11855726856691, 'MSW': 0.00036627685671318484, 'df_between': 16, 'df_within': 20808175, 'F': 131372.0929582139, 'p_value': 1.1102230246251565e-16}

ANOVA Conjunto 2 (2020-01-01 a 2020-09-22)
{'k': 19, 'N': 10980132, 'SSB': 395.8281673458155, 'SSW': 2900.5213585814513, 'MSB': 21.990453741434195, 'MSW': 0.00026416133955829517, 'df_between': 18, 'df_within': 10980113, 'F': 83246.29856210029, 'p_value': 1.1102230246251565e-16}


#### Interpretación
- **Hipótesis nula (H0)**: las medias de `price` son iguales entre niveles de `product_group_name`.
- **Hipótesis alternativa (H1)**: al menos una media difiere.
- **Criterio**: rechazar H0 si p-valor < 0.05.

A continuación, se muestra un reporte resumido para cada conjunto.


In [8]:
alpha = 0.05

def print_report(nombre, res):
    print(f"\n=== {nombre} ===")
    print(f"k (grupos): {res['k']}")
    print(f"N (observaciones): {res['N']:,}")
    print(f"df_between: {res['df_between']}, df_within: {res['df_within']}")
    print(f"SSB: {res['SSB']:.6f}, SSW: {res['SSW']:.6f}")
    print(f"MSB: {res['MSB']:.6f}, MSW: {res['MSW']:.6f}")
    print(f"F: {res['F']:.6f}, p-valor: {res['p_value']}")
    if math.isfinite(res['p_value']) and res['p_value'] < alpha:
        print("Decision: Rechazar H0 (diferencias significativas entre medias por grupo)")
    else:
        print("Decision: No rechazar H0 (no evidencia suficiente de diferencias)")

print_report("Conjunto 1 (2018-09-20 a 2019-12-31)", res1)
print_report("Conjunto 2 (2020-01-01 a 2020-09-22)", res2)



=== Conjunto 1 (2018-09-20 a 2019-12-31) ===
k (grupos): 17
N (observaciones): 20,808,192
df_between: 16, df_within: 20808175
SSB: 769.896916, SSW: 7621.552933
MSB: 48.118557, MSW: 0.000366
F: 131372.092958, p-valor: 1.1102230246251565e-16
Decision: Rechazar H0 (diferencias significativas entre medias por grupo)

=== Conjunto 2 (2020-01-01 a 2020-09-22) ===
k (grupos): 19
N (observaciones): 10,980,132
df_between: 18, df_within: 10980113
SSB: 395.828167, SSW: 2900.521359
MSB: 21.990454, MSW: 0.000264
F: 83246.298562, p-valor: 1.1102230246251565e-16
Decision: Rechazar H0 (diferencias significativas entre medias por grupo)
